# Notebook 01: CES Model Calibration and Fit

**Project:** Brent Crude Oil Prices and Renewable Energy Investment  
**Date:** 2026-04-06  

This notebook calibrates the CES substitution model to observed data and
validates the fit against the Mukhtarov et al. (2024) empirical elasticity.

## Key Parameters

| Parameter | Value | Source |
|---|---|---|
| Substitution elasticity sigma | 1.8 | Papageorgiou et al. 2017 |
| Oil price elasticity | +0.16% per 1% | Mukhtarov et al. 2024 |
| Renewable share 2024 | ~30% of new capacity | IEA WEI 2025 |
| Base investment 2024 | $807 bn | IEA/IRENA |
| Brent long-run mean | $57.46/bbl | EIA RBRTE |


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from ces_model.core.ces import CESModel, ces_substitution, DEFAULT_SIGMA, DEFAULT_ALPHA
from ces_model.data.eia import load_investment_series
from ces_model.data.cleaning import annualise_brent

print(f"Default sigma = {DEFAULT_SIGMA}  (Papageorgiou et al. 2017)")
print(f"Default alpha = {DEFAULT_ALPHA}  (calibrated to 2024 renewable share)")

## 1. CES Share Function: Price Sensitivity

Plot renewable share as a function of the oil-to-renewable price ratio
for three values of sigma.

In [ ]:
# Sweep oil price from 10 to 200 USD/bbl; renew price fixed at 1.0 (normalised)
oil_prices = np.linspace(10.0, 200.0, 300)
sigmas = [1.0, 1.8, 3.0]
alpha = DEFAULT_ALPHA

fig, ax = plt.subplots(figsize=(8, 5))
for s in sigmas:
    shares = [
        ces_substitution(p, 1.0, sigma=s, alpha=alpha).renew_share
        for p in oil_prices
    ]
    label = f"sigma={s}" + (" (Papageorgiou 2017)" if s == 1.8 else "")
    ax.plot(oil_prices, shares, label=label)

ax.axhline(alpha, color="gray", linestyle="--", linewidth=0.8, label=f"alpha={alpha} (parity)")
ax.axvline(57.46, color="orange", linestyle=":", linewidth=0.8, label="Long-run mean $57.46/bbl")
ax.set_xlabel("Brent Oil Price (USD/bbl, renew price index = 1.0)")
ax.set_ylabel("Renewable Energy Share")
ax.set_title("CES Share Function: Renewable Share vs Oil Price")
ax.legend()
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figures/01a_ces_share_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figures/01a_ces_share_curve.png")

## 2. Alpha Calibration

Calibrate alpha so the model matches an observed renewable share at a given
reference oil price. We use the 2024 STEPS baseline ($80/bbl) and a
30% renewable share of new capacity.

In [ ]:
# Calibration target: 30% renewable share at $80/bbl (2024 actual)
model = CESModel(sigma=DEFAULT_SIGMA)
model.calibrate_alpha(observed_share=0.30, price_oil=80.0, price_renew=1.0)

result = model.compute_share(price_oil=80.0)
print(f"Calibrated alpha: {model.alpha:.6f}")
print(f"Verified share at $80/bbl: {result.renew_share:.6f}  (target: 0.30)")

# Cross-check against Mukhtarov et al. (2024) elasticity
validation = model.validate_mukhtarov(base_price=65.0)
print(f"\nMukhtarov validation (base $65/bbl):")
print(f"  Implied elasticity: {validation['implied_elasticity']:.4f}")
print(f"  Mukhtarov target:   {validation['mukhtarov_target']:.4f}")
print(f"  Ratio (model/target): {validation['ratio']:.2f}x")

## 3. Sensitivity of Share to Sigma

Sweep sigma from 0.5 to 4.0 and compute the renewable share at a
high oil price ($100/bbl) to show sensitivity to the elasticity parameter.

In [ ]:
sigma_range = np.linspace(0.5, 4.0, 200)
share_at_100 = [
    ces_substitution(100.0, 1.0, sigma=s, alpha=model.alpha).renew_share
    for s in sigma_range
]
share_at_50 = [
    ces_substitution(50.0, 1.0, sigma=s, alpha=model.alpha).renew_share
    for s in sigma_range
]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sigma_range, share_at_100, label="Oil price = $100/bbl (high)")
ax.plot(sigma_range, share_at_50, label="Oil price = $50/bbl (low)")
ax.axhline(model.alpha, color="gray", linestyle="--", linewidth=0.8,
           label=f"alpha = {model.alpha:.3f}")
ax.axvline(DEFAULT_SIGMA, color="steelblue", linestyle=":", linewidth=1.0,
           label=f"sigma = {DEFAULT_SIGMA} (Papageorgiou 2017)")
ax.set_xlabel("Sigma (elasticity of substitution)")
ax.set_ylabel("Renewable Energy Share")
ax.set_title("Renewable Share vs Sigma at Two Oil Price Levels")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figures/01b_sigma_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Historical Brent vs Investment: Correlation Context

Load the IEA/IRENA investment series and overlay the annual Brent average
to visualise the data used for calibration.

In [ ]:
invest_df = load_investment_series()
print(invest_df.to_string(index=False))

In [ ]:
# Annual Brent averages 2015-2024 (from EIA RBRTE, calibrated to data analyst findings)
brent_annual = pd.DataFrame({
    "year": list(range(2015, 2025)),
    "brent_avg": [52.4, 43.6, 54.2, 71.3, 64.4, 41.7, 70.9, 101.0, 82.5, 80.0]
})

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.bar(invest_df["year"], invest_df["renewable_power_invest_bn_usd"],
        color="steelblue", alpha=0.7, label="Renewable investment (USD bn)")
ax2.plot(brent_annual["year"], brent_annual["brent_avg"],
         color="darkorange", marker="o", linewidth=2, label="Brent avg (USD/bbl)")

ax1.set_xlabel("Year")
ax1.set_ylabel("Renewable Investment (USD bn)", color="steelblue")
ax2.set_ylabel("Brent Avg Price (USD/bbl)", color="darkorange")
ax1.set_title("Renewable Power Investment vs Brent Annual Average (2015-2024)")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
ax1.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figures/01c_invest_vs_brent_calibration.png", dpi=150, bbox_inches="tight")
plt.show()

corr = invest_df["renewable_power_invest_bn_usd"].corr(brent_annual["brent_avg"])
print(f"Pearson r (renewable invest vs Brent annual avg): {corr:.3f}")
print("Note: Positive correlation consistent with Mukhtarov et al. +0.16% elasticity.")
print("Caution: time-trend confounding with n=10; see data analysis for caveats.")

## 5. Summary: Calibrated Model Parameters

The calibrated model is ready for scenario analysis (Notebook 02) and
sensitivity analysis (Notebook 03).

In [ ]:
print("=" * 55)
print("CALIBRATED CES MODEL -- SUMMARY")
print("=" * 55)
print(f"  sigma (EOS):          {model.sigma}  (Papageorgiou et al. 2017)")
print(f"  alpha (renew weight): {model.alpha:.6f}  (calibrated to 30% share at $80/bbl)")
print(f"  oil_elasticity:       {model.oil_elasticity}  (Mukhtarov et al. 2024)")
print(f"  base_invest (2024):   $807 bn  (IEA/IRENA)")
print()

# Show shares at key price levels
print("Renewable share at key Brent price points:")
for price in [25, 44, 65, 80, 100, 130]:
    r = model.compute_share(float(price))
    print(f"  ${price:3d}/bbl -> {r.renew_share:.1%}")